# Qwen3.5-0.8B로 Jev-like 의사결정 모델 만들기

**목표:** 각자의 환경(CPU, NVIDIA CUDA, Apple Silicon 등)에서 Qwen3.5-0.8B를 미세조정해, 입력 상태와 질문을 받아 선택지·참/거짓·척도에 대한 확률을 반환하는 모델을 만든다. 데이터는 BANKING77(Choice), GoEmotions(Noul), SST-5(Score)를 사용한다.

이 노트북의 구성:

1. **Jev like 모델 구현 방법론** — 관련 연구를 분류하고 각각의 입력, readout, 손실, 장단점과 실습 적용 방법을 설명한다.
2. **Jev like 모델 학습에 사용할 데이터셋** — 데이터 이해, 표본 추출, typed question 변환과 누출 없는 분할.
3. **Jev like 모델을 학습하는 pipeline** — 공통 데이터·평가 계약과 PyTorch/MLX별 모델, 학습, 확률 보정과 체크포인트.
4. **Jev like 모델 경량화** — 4-bit / 8-bit quantization과 정확도·확률·메모리 변화.
5. **Jev like 모델 배포 및 사용** — 로컬 typed decision API와 애플리케이션 연결.

**환경 선택:** PyTorch CPU/CUDA를 공통 경로로, Apple Silicon MLX를 대안 경로로 설계한다. 1.14에서 환경별 실습 규모와 확인 사항을 고른다. 같은 데이터·입출력·loss·평가를 사용하되 backend별 구현과 저장 형식은 구분한다.

**현재 작성 범위:** 1번 방법론을 완성하고, 2번에는 다운로드한 원본의 위치와 현황을 기록한다. 2~5번의 상세 실습은 이어서 작성한다. 모델 가중치 다운로드·학습·양자화·서버 실행은 아직 수행하지 않았다.

**자료 확인 기준: 기존 분류 2026-10-01, Laya·decider calibration 분석(1.11.1~1.11.7) 2026-10-03.** 아래 분류는 이 노트북이 학습을 위해 구성한 분류 체계이다. 학술 논문, arXiv preprint, 저자 기술 보고, 공개 코드와 제품 발표를 구분한다. 공개 대안의 구조를 TypeSafe Jev의 내부 구조로 해석하지 않는다. [R1](https://typesafe.ai/blog/introducing-system-one-models-and-jev)

## 1. Jev like 모델 구현 방법론

### 1.1 목적과 입출력 계약

일반 LLM은 다음 토큰을 반복 생성한다. 의사결정 모델은 **소프트웨어가 필요한 유한한 판단을 확률로 제공**한다. 예를 들어 문의를 부서로 보내거나, 감정이 포함됐는지 판단하거나, 장애 정도를 정해진 척도로 평가한다. 형식이 고정되어도 정답과 확률의 신뢰성은 별도로 학습·검증해야 한다. TypeSafe는 이 계약을 `state + typed questions → probabilistic answers`로 설명한다. [R1](https://typesafe.ai/blog/introducing-system-one-models-and-jev)

| primitive | 질문 | 출력 | 수학적 해석 |
| --- | --- | --- | --- |
| Choice | 이 문의의 주된 의도는? | 후보별 확률과 선택한 키 | 배타적 범주 분포: $\sum_k p_k=1$ |
| Noul | 분노가 포함되어 있는가? | $P(\mathrm{true})$ | 하나의 명제에 대한 Bernoulli 분포 |
| Score | 정해진 척도에서 어느 수준인가? | level별 분포와 기대 점수 | 순서가 있는 범주 분포: $E[v]=\sum_k v_kp_k$ |

Score의 기대값이 1.7이라는 것은 level 1과 2 등에 분포한 판단을 요약한 것이다. 그것이 “감정 강도 1.7”이라는 의미가 되려면 정답 데이터와 척도 정의가 필요하다. **GoEmotions의 감정 태그만으로 강도 정답을 만들지는 않는다.** Score 실습에는 별도의 SST-5 5단계 감성 극성 라벨을 사용한다(2.1 참조).

`confidence`, 최고 후보 확률, entropy는 서로 다른 값이다. API별 confidence 정의를 그대로 정답률로 해석하지 않는다. 이 실습은 후보 확률과 실측 정답률을 비교하고, 자동 처리·사람 검토 임계값을 별도 validation에서 정한다.

### 1.2 분류의 기준: 점수 계산 구조와 학습 목표를 구분하기

**출처를 읽는 기준.** 아래 각 후보의 “원리의 배경과 파생 관계”는 (a) 기반 아이디어를 설명하는 연구, (b) 저자가 명시한 직접 파생 관계, (c) 이번 노트북의 응용 설계를 구분한다. 아이디어가 비슷하다는 이유만으로 저장소가 특정 논문에서 직접 파생됐다고 주장하지 않는다. 인용은 해당 설명 바로 옆의 저자·연도와 원문 링크로 연결한다.

구현을 먼저 **점수를 어디서 읽는가**로 나눈다. 이후 같은 구조에 SFT, contrastive learning, distillation, calibration-aware RL 같은 학습 목표를 적용할 수 있다. LoRA는 사전학습 가중치를 고정하고 저랭크 업데이트를 학습하는 Hu et al.(2021)의 방법이며 readout 구조를 정하는 방법은 아니다. [R25](https://arxiv.org/abs/2106.09685) 4/8-bit quantization 역시 가중치 표현과 실행에 관한 별도 축이다.

| 구분 | 구현 방법 | 입력 구성 / 점수 위치 | 유리한 경우 | 대표 자료 |
| --- | --- | --- | --- | --- |
| A | 고정 라벨 classification head | state 표현 → 정해진 $K$개 head 출력 | 하나의 업무·라벨 체계 | BERT 계열 supervised classifier [R17](https://arxiv.org/abs/1810.04805v2) |
| B | answer-token / code readout | state·질문·모든 후보 → 마지막 입력 위치 | Qwen 활용, 후보 설명을 바꿀 수 있는 계약 | Jeff 공개 구현, decision-token 연구 [R3](https://github.com/firelex/jeff/blob/f06788292874c21a5b5c41549ac220dd9e15da7f/src/jeff/model.py), [R9](https://arxiv.org/abs/2601.13284v1) |
| C | 후보별 scalar scoring | state·질문·후보를 각각 encode → 공용 scalar head | 후보 수가 가변, 후보별 자세한 비교 | NLI 기반 분류 [R18](https://arxiv.org/abs/1909.00161v1), [R4](https://arxiv.org/abs/2312.17543v1), Kev·NanoJev [R5](https://github.com/jaredpalmer/kev/blob/main/kev/model.py), [R6](https://github.com/TianyuCodings/NanoJev/tree/76fdfc9ecdca45a9bcef17991a07d3041a87685a) |
| D | label-aware joint encoding | state와 모든 label을 함께 encode → label별 scorer | 한 번의 입력 처리로 여러 라벨 판단 | GLiNER → GLiClass → SCX Router [R19](https://arxiv.org/abs/2311.08526), [R7](https://arxiv.org/abs/2508.07662v1), [R8](https://arxiv.org/abs/2609.02292v1) |
| E | dual-encoder contrastive scoring | state와 action을 따로 encode → 내적 | 후보 재사용·대규모 action 검색 | Sentence-BERT·InfoNCE 계열, CLM [R20](https://arxiv.org/abs/1908.10084v1), [R21](https://arxiv.org/abs/1807.03748v2), [R11](https://github.com/Contrastive-LM/CLM) |
| F | diffusion answer-slot readout | 고정 answer canvas의 slot별 분포 | 여러 slot을 병렬로 읽는 실험 | discrete diffusion 계열, djev [R22](https://arxiv.org/abs/2107.03006), [R23](https://arxiv.org/abs/2502.09992), [R12](https://github.com/mmastrac/djev) |

이 표의 “한 번”은 모델 호출 수를 뜻할 수도, state 계산을 한 번만 한다는 뜻일 수도 있다. **호출 1회·batch 1개·state prefix 계산 1회·전체 작업 완료 1회는 각각 따로 측정한다.** 후보별 경로를 batch로 묶어도 state를 후보마다 계산하면 연산량은 후보 수에 따라 커진다.

**그림 읽는 법.** 아래 Mermaid 흐름도에서 실선은 데이터·제어 흐름이고, 점선은 명시한 학습 업데이트·반복·cache 재사용 경로다. 학습 노드는 실제 학습 시에만 사용한다. A~F는 점수를 읽는 구조를, 1.9~1.12는 학습 목표와 보정을 표현한다. 세부 구현은 본문과 인용한 소스를 함께 확인한다.

Mermaid 렌더링을 지원하는 notebook 뷰어에서 그림으로 볼 수 있다. 지원하지 않는 뷰어에서는 코드 블록으로 표시되며, 블록 내용을 Mermaid 지원 에디터에 붙여 넣어 확인할 수 있다.


### 1.3 A — 고정 라벨 classification head

**원리의 배경과 파생 관계.** 사전학습 Transformer 표현에 작은 task head를 붙여 지도학습하는 대표적 근거는 Devlin et al.(2018)의 *BERT*, §3 및 §4.1이다. BERT는 `[CLS]` 표현을 분류에 사용한다. 이 실습은 그 일반적인 “표현 + 분류층” 패턴을 causal Qwen의 마지막 유효 토큰에 적용하는 설계이며, BERT의 pooling이나 attention을 그대로 재현하는 것은 아니다. [R17](https://arxiv.org/abs/1810.04805v2)

**원리.** 토큰화한 문장을 Qwen backbone으로 처리하고 마지막 유효 입력 토큰의 표현 $h$에 선형층을 붙인다.

$$z=Wh+b,\qquad p_k=\operatorname{softmax}(z)_k.$$

BANKING77은 77개 head 출력 중 정답 하나에 cross entropy를 적용할 수 있다. GoEmotions는 28개 출력 각각에 sigmoid를 적용하고 multi-hot 정답에 binary cross entropy를 적용한다. GoEmotions에서 동시에 나타날 수 있는 감정들을 softmax로 경쟁시키지 않는다.

**학습 대상.** 첫 단계는 backbone을 고정하고 head만 학습한다. 이후 backbone 일부에 LoRA를 적용하고 head와 함께 업데이트할 수 있다. 이 단계는 학습되는 파라미터와 loss를 쉽게 확인할 수 있는 비교 기준이다.

**프로세스 — 고정 라벨 head**

```mermaid
flowchart TD
    S["입력 문장 state"] --> TOK["Tokenizer와 attention mask"]
    TOK --> BB["Qwen backbone"]
    BB --> H["마지막 유효 토큰 표현 h"]
    H --> HEAD["고정 K-label 선형 head"]
    HEAD --> Z["K개 logits"]
    Z --> CH["BANKING77: softmax"]
    Z --> NO["GoEmotions: label별 sigmoid"]
    CH --> OUT1["배타적인 의도 확률과 선택 라벨"]
    NO --> OUT2["독립적인 감정 확률"]
    Z --> L["학습: 단일 라벨 CE 또는 multi-label BCE"]
    Y["정답 라벨 또는 multi-hot vector"] --> L
    L --> OPT["Backward와 optimizer"]
    OPT -.->|학습 시 업데이트| HEAD
```

고정 라벨 head는 후보 설명을 입력받지 않는다. 첫 실습에서는 backbone을 고정하고 head만 학습한다. LoRA 단계에서는 adapter도 업데이트 대상에 포함한다. CE/BCE는 확률을 다시 log로 변환하기보다 logits에서 안정적으로 계산한다.

**장점.** 후보 설명을 매번 입력하지 않아 짧고 간단하며, 정해진 태스크의 baseline을 만들기 좋다.

**한계.** head의 각 행은 특정 라벨에 연결된다. 새 라벨·새 판단 기준을 받으려면 head와 데이터를 바꾸어야 한다. 이 baseline의 test 정확도가 높아도 임의 질문이나 보지 못한 라벨에 대한 능력을 입증하지는 않는다.

**실습 적용.** 고정 BANKING77 subset 및 고정 감정 태그의 baseline으로 사용한다. 이후 B/C 방식과 같은 test를 비교해, 후보 설명을 입력하는 비용이 어떤 효과를 주는지 평가한다. 위 수식과 적용 순서는 이 노트북의 구현 설계이며 특정 Jev 구조의 재현 주장이 아니다.

### 1.4 B — answer-token / code readout: Qwen에서 시작하기

**원리의 배경과 파생 관계.** autoregressive LM의 다음 토큰 분포에서 답에 대응하는 토큰 점수를 읽는 방식이다. decision-token 확률과 보정을 연구한 Yaldiz et al.(2026)은 이 readout을 이해하는 연구 근거다. [R9](https://arxiv.org/abs/2601.13284v1) **직접적인 코드 계보**는 Jeff README에 명시된 AutoJev → Jeff이며, 별도 code readout 초기화는 Jeff `DecisionModel` 소스를 인용한다. 두 저장소는 TypeSafe Jev와 독립적인 공개 구현이다. Yaldiz 논문이 Jeff의 직접 기반이라고 확인된 것은 아니다. [R29](https://github.com/denis-pplx/autojev), [R3](https://github.com/firelex/jeff/blob/f06788292874c21a5b5c41549ac220dd9e15da7f/src/jeff/model.py)

**원리.** 질문과 후보 설명을 프롬프트에 넣고 후보를 단일 토큰 코드에 연결한다. 예를 들어 `A=이중 결제`, `B=환불 요청`, `C=카드 분실`을 입력한다. 입력을 처리한 마지막 위치의 LM logits에서 코드에 해당하는 행만 읽는다.

$$z_k=\mathrm{LMHead}(h_{\mathrm{last}})_{\mathrm{token}(code_k)},\qquad
p_k=\frac{\exp(z_k/T)}{\sum_{j\in\mathcal C}\exp(z_j/T)}.$$

이 확률은 **제시한 후보 집합 내 조건부 확률**이다. 전체 vocabulary에서 A의 확률과 같지 않다. 후보 목록을 바꾸면 분모와 판단이 달라질 수 있다. 코드가 단일 토큰인지, 실제 chat prefix 뒤에서도 동일한 토큰으로 붙는지 확인해야 한다. [R3](https://github.com/firelex/jeff/blob/f06788292874c21a5b5c41549ac220dd9e15da7f/src/jeff/model.py), [R9](https://arxiv.org/abs/2601.13284v1)

**Jeff의 구체적 구현.** 공개 `DecisionModel`은 선택지 코드의 LM head 행으로 별도 readout을 초기화하고, backbone과 readout을 전체 가중치 SFT로 업데이트한다. 추론은 유효 후보 밖의 logits를 mask하고 temperature를 적용한다. 코드를 문자열로 출력하지 않고 점수에서 선택 키로 매핑한다. [R3](https://github.com/firelex/jeff/blob/f06788292874c21a5b5c41549ac220dd9e15da7f/src/jeff/model.py)

**프로세스 — Jeff식 code readout 실습 경로**

```mermaid
flowchart TD
    S["state와 question"] --> PROMPT["모든 후보 설명과 code를 넣은 prompt"]
    C["후보 키와 단일 토큰 code 매핑"] --> PROMPT
    PROMPT --> TOK["Chat template와 tokenizer"]
    TOK --> BB["Qwen backbone: 입력 prefill"]
    BB --> H["마지막 유효 입력 위치의 h"]
    INIT["사전학습 LM head의 code-token 행"] -->|초기화| HEAD["별도 학습 가능한 code readout"]
    H --> HEAD
    HEAD --> MASK["현재 후보 밖의 logits mask"]
    C --> MASK
    MASK --> TEMP["추론: temperature 적용"]
    TEMP --> P["유효 후보에 대한 softmax"]
    P --> OUT["code를 원래 후보 키로 매핑한 typed answer"]
    C --> OUT
    MASK --> LOSS["학습: 정답 code에 대한 CE"]
    Y["정답 후보 키를 code index로 변환"] --> LOSS
    LOSS --> OPT["Readout-only 또는 LoRA와 readout 업데이트"]
    OPT -.->|학습 시 업데이트| HEAD
```

그림은 LM head 행으로 **별도 readout을 초기화하는 경로**다. 미세조정 전 baseline은 원래 LM logits에서 code 행을 직접 읽는다. Noul을 false/true 두 code로 구성하면 true 후보 확률을 반환한다. Score는 level별 확률을 기대 점수로 요약한다. 문자열 답변 생성·파싱 단계는 필요하지 않다.

**장점.** Qwen의 사전학습 표현과 code-token 방향을 활용한다. 후보를 자연어로 설명할 수 있어 typed question 인터페이스에 연결하기 쉽다.

**한계.** 코드 선호·후보 순서 편향과 학습 중 본 후보 수의 영향이 남는다. 설명을 받는 인터페이스가 새 라벨에 대한 일반화를 자동 보장하지 않는다. 후보 설명과 코드를 섞어 학습하되 정답 매핑도 함께 갱신해야 한다.

**실습 적용.** 첫 Jev-like 경로로 B를 선택한다. BANKING77은 코드 중 하나를 선택하고, GoEmotions는 감정별 독립 `false/true` 질문으로 구성한다. SST-5는 부정→긍정 순서의 5개 level을 코드에 연결해 Score 분포를 학습한다. PyTorch와 MLX 모두 readout-only로 먼저 확인한 뒤 LoRA+readout을 학습하는 순서로 설계한다. 이는 Jeff의 full-weight recipe를 실습 규모에 맞게 변형하는 실험이다. LoRA의 근거는 Hu et al.(2021)이다. [R25](https://arxiv.org/abs/2106.09685)

### 1.5 C — 후보별 scalar scoring / cross-encoder

**원리의 배경과 파생 관계.** Yin, Hay & Roth(2019)은 zero-shot 분류를 text–label entailment 문제로 구성했고(§3), Laurer et al.(2023)은 NLI 기반 범용 분류를 확장했다. [R18](https://arxiv.org/abs/1909.00161v1), [R4](https://arxiv.org/abs/2312.17543v1) 여기서 가져오는 원리는 **입력 문장과 후보 설명을 함께 처리해 관련성을 점수화**하는 것이다. Kev·NanoJev는 후보별 scalar scoring의 공개 구현 근거다. 아래 공용 head·question 단위 CE는 본 실습의 설계이며, 이 저장소들이 Yin/Laurer 코드를 직접 계승했다는 주장은 아니다. [R5](https://github.com/jaredpalmer/kev/blob/main/kev/model.py), [R6](https://github.com/TianyuCodings/NanoJev/tree/76fdfc9ecdca45a9bcef17991a07d3041a87685a)

**원리.** 후보마다 `(state, question, candidate_description)` 입력 경로를 만들고 모든 경로에 같은 모델과 scalar head를 적용한다.

$$h_k=f_\theta(s,q,c_k),\qquad z_k=w^Th_k+b.$$

Choice는 전체 후보 점수에 softmax와 question 단위 CE를 적용한다. Noul은 하나의 명제 logit에 sigmoid+BCE를 적용하거나 false/true 두 후보를 비교한다. Score는 level 후보를 평가한다. 후보 scalar를 개별 정답처럼만 학습하면 후보 간 경쟁을 충분히 학습하지 못할 수 있다.

**관련 흐름.** NLI는 입력 문장을 premise, 라벨 설명을 hypothesis로 만들어 entailment를 평가한다. Laurer 등의 universal classification 연구가 이 경로를 다룬다. Kev와 NanoJev도 후보별 점수 계산을 공개하며, head와 입력 계약은 각 구현에서 확인해야 한다. [R4](https://arxiv.org/abs/2312.17543v1), [R5](https://github.com/jaredpalmer/kev/blob/main/kev/model.py), [R6](https://github.com/TianyuCodings/NanoJev/tree/76fdfc9ecdca45a9bcef17991a07d3041a87685a)

**프로세스 — 후보별 cross-encoder 점수와 Choice**

```mermaid
flowchart TD
    INPUT["state, question, 후보 목록"] --> PAIRS["후보마다 입력을 구성: K개 경로"]
    PAIRS --> P1["state + question + 후보 1"]
    PAIRS --> P2["state + question + 후보 2"]
    PAIRS --> PK["state + question + 후보 K"]
    P1 --> ENC["같은 encoder와 scalar head를 각 경로에 적용"]
    P2 --> ENC
    PK --> ENC
    ENC --> Z["후보당 scalar 하나: z1부터 zK"]
    Z --> P["질문 내 후보 전체에 softmax"]
    P --> OUT["후보 확률과 선택 키"]
    Z --> L["학습: 질문 단위 CE"]
    Y["정답 후보 index"] --> L
    L --> OPT["공용 head와 학습 대상 adapter 업데이트"]
    OPT -.->|학습 시 업데이트| ENC
```

encoder와 head의 **가중치는 공유**하지만, 그림의 세 입력은 각각 계산하는 경로다. batch로 묶어도 공통 state 계산이 자동으로 공유되지는 않는다. 단일 명제 Noul은 scalar logit에 sigmoid+BCE를 사용하거나 false/true 두 후보를 구성한다.

**장점.** head 크기가 후보 수에 고정되지 않고, state와 후보의 의미적 관계를 상세히 처리한다.

**한계.** $K$개 후보의 입력을 반복 처리한다. batch 호출 1회여도 공통 prefix 재사용이 없다면 $K$개 경로 연산이 필요하다. 후보별 연산을 독립 처리하면 후보 사이 비교 정보를 얻는 정도가 제한된다.

**실습 적용.** BANKING77의 적은 후보 subset에서 B와 비교하기 좋다. GoEmotions는 “이 문장에 anger가 표현되어 있는가?”처럼 yes/no 명제로 변환할 수 있다. 감정별 negative가 너무 많아지지 않도록 표본 비율과 negative 선택을 따로 기록한다.

### 1.6 D — label-aware joint encoding: GLiClass와 SCX Router

**원리의 배경과 파생 관계.** 이 후보는 저자가 명시한 파생 관계를 따라갈 수 있다. Zaratiana et al.(2023)의 **GLiNER**는 자연어 entity label과 text를 함께 표현한다. [R19](https://arxiv.org/abs/2311.08526) Stepanov et al.(2025)의 **GLiClass**는 GLiNER의 uni-encoder 설계를 sequence classification에 적용한다고 명시한다(Introduction, §2.1). [R7](https://arxiv.org/abs/2508.07662v1) **SCX Router**(2026)는 GLiClass를 Qwen3 causal decoder와 label suffix의 bidirectional scorer로 확장한다(§2.2, §4). [R8](https://arxiv.org/abs/2609.02292v1) 아래 수식은 scorer의 일반화된 설명이며 각 논문의 모든 변형을 동일한 내적 모델로 축약한 것은 아니다.

**원리.** state와 label 설명을 같은 입력에 넣어 text 표현과 label 표현을 함께 만든다. label별 marker·span에서 표현 $v_k$를 얻고 pooled text 표현 $u$와 scorer를 적용한다.

$$z_k=g(u,v_k),\quad \text{예: }g(u,v_k)=u^Tv_k/\tau.$$

GLiClass의 주된 uni-encoder는 label과 text가 상호작용하도록 설계되어 있다. 논문은 bi-encoder, fused bi-encoder 등 변형도 설명하므로 모든 GLiClass 변형을 단일 구조로 단정하지 않는다. 단일 라벨 선택과 다중 라벨 감지에는 각각 다른 정규화·loss를 적용한다. [R7](https://arxiv.org/abs/2508.07662v1)

**최근 연구.** SCX Router(2026-09)는 Qwen3 decoder와 얕은 bidirectional scorer를 결합하고 대화의 text-only KV cache를 재사용하는 routing 경로를 제안한다. 후보 label을 영구 cache에 남기지 않는 점이 특징이다. 이는 Qwen3.5-0.8B에 바로 적용된다는 증거가 아니라 decoder 기반 동적 라벨 분류의 설계 참고이다. [R8](https://arxiv.org/abs/2609.02292v1)

**프로세스 ① — GLiClass의 uni-encoder 개념**

```mermaid
flowchart TD
    S["입력 text"] --> JOIN["Label marker와 모든 label 설명을 text와 결합"]
    LABELS["자연어 label 목록"] --> JOIN
    JOIN --> ENC["Bidirectional joint encoder"]
    ENC --> U["Text pooling: 표현 u"]
    ENC --> V["Label pooling: 표현 v1부터 vK"]
    U --> SC["공용 text-label scorer"]
    V --> SC
    SC --> Z["Label별 logits"]
    Z --> ONE["단일 라벨: softmax와 CE"]
    Z --> MULTI["다중 라벨: sigmoid와 BCE"]
    ONE --> OUT["Label별 판단"]
    MULTI --> OUT
```

GLiClass의 대표적 uni-encoder를 요약한 그림이다. label과 text가 encoder 안에서 상호작용하며, pooling/scorer의 구체적 구현은 변형마다 다르다. label마다 text 전체를 따로 encode하는 C와 입력 구성이 다르다.

**프로세스 ② — SCX Router의 decoder-KV 경로**

```mermaid
flowchart TD
    TURN["새 대화 turn의 text tokens"] --> DEC["Qwen3 causal decoder"]
    CACHE["이전 대화의 text-only KV cache"] --> DEC
    DEC --> NEWCACHE["새 text까지 포함한 KV cache"]
    NEWCACHE -.->|다음 turn에서 재사용| CACHE
    NEWCACHE --> SUFFIX["후보 label suffix를 임시로 처리"]
    LABELS["현재 후보 label 목록"] --> SUFFIX
    SUFFIX --> BIDI["얕은 bidirectional label scorer"]
    BIDI --> Z["후보별 suitability logits"]
    Z --> P["태스크에 맞는 sigmoid 또는 softmax"]
    P --> POLICY["비용과 자격 조건을 반영한 routing policy"]
    POLICY --> OUT["선택한 모델 또는 판단 결과"]
    SUFFIX --> DROP["후보 label의 임시 KV는 폐기"]
```

label KV는 영구 대화 cache에 추가하지 않는다. 이 그림은 GLiClass를 decoder와 cache 구조로 확장한 **SCX 설계의 참고도**이며 Qwen3.5 실습의 구현 완료를 뜻하지 않는다. sigmoid suitability와 최종 routing policy도 구분한다.

**장점.** 후보마다 state 전체를 다시 넣는 경로를 줄이고 label 간 관계를 활용할 수 있다.

**한계.** label 수가 늘면 입력 길이와 scorer 비용도 늘어난다. label 위치에 따라 볼 수 있는 정보가 달라진다. **causal Qwen에 GLiClass 입력 문자열만 복사하면 bidirectional uni-encoder와 같은 동작을 얻지 못한다.** attention mask·pooling·label scorer를 함께 설계해야 한다.

**실습 적용.** 이 노트북의 초기 구현보다 복잡하므로 B/C 이후의 확장 후보로 둔다. 먼저 입력 길이, label별 표현의 접근 범위, 후보 순서 변경 효과를 확인한다.

### 1.7 E — dual-encoder contrastive scoring: CLM

**원리의 배경과 파생 관계.** Reimers & Gurevych(2019)의 *Sentence-BERT*는 문장을 각각 encode해 similarity를 계산하는 siamese/bi-encoder 계열의 대표적 근거다. [R20](https://arxiv.org/abs/1908.10084v1) van den Oord et al.(2018)의 *Contrastive Predictive Coding*, §2.3은 positive를 negative 집합에서 구분하는 **InfoNCE** 목적의 근거다. [R21](https://arxiv.org/abs/1807.03748v2) CLM은 이 계열의 state–action embedding과 contrastive decision learning을 보여주는 최근 기술 보고·구현으로 인용한다. 두 기반 논문은 구조와 loss의 개념적 배경이며, CLM이 Sentence-BERT 코드를 직접 fork했다는 뜻은 아니다. [R11](https://github.com/Contrastive-LM/CLM)

**원리.** state와 action을 분리해 encode한 뒤 같은 공간으로 projection한다.

$$u=\operatorname{norm}(P_s f(s,q)),\quad
v_k=\operatorname{norm}(P_a f(c_k)),\quad z_k=u^Tv_k/\tau.$$

정답 state–action pair를 가까이, 다른 pair를 멀리 만드는 InfoNCE 계열 loss를 사용한다. 다음은 이 노트북의 단방향 예시이며 CLM 전체 recipe의 동일 구현이라는 뜻은 아니다.

$$L_i=-\log\frac{\exp(u_i^Tv_i/\tau)}{\sum_j\exp(u_i^Tv_j/\tau)}.$$

**CLM 자료의 성격.** *Contrastive Language Models: A System One Model for Fast and Generalizable Decision-Making*은 공개 저장소에서 저자 기술 보고/Notion Blog로 인용되어 있다. 이 노트북에서는 정식 arXiv 논문으로 표기하지 않는다. 공개 fine-tuning 경로는 Qwen3-8B embedding 위 projection heads 등을 학습한다. [R11](https://github.com/Contrastive-LM/CLM)

**프로세스 ① — dual-encoder 추론과 후보 cache**

```mermaid
flowchart TD
    STATE["현재 state와 question"] --> SE["State encoder"]
    SE --> SP["State projection과 정규화: u"]
    ACTIONS["재사용할 후보 action 설명"] --> AE["Action encoder"]
    AE --> AP["Action projection과 정규화: v1부터 vK"]
    AP --> CACHE["Action embedding cache"]
    SP --> SIM["후보마다 내적 계산과 temperature 적용"]
    CACHE --> SIM
    SIM --> P["현재 후보 집합에서 softmax"]
    P --> OUT["후보 확률과 선택 action"]
```

후보·encoder·projection 가중치가 그대로일 때 action embedding을 재사용할 수 있다. state와 후보 사이의 token-level cross attention은 이 경로에 없다. 가중치나 후보 설명이 바뀌면 cache를 다시 만든다.

**프로세스 ② — contrastive 학습**

```mermaid
flowchart TD
    PAIRS["정답 state-action pair batch"] --> S["State encoder와 projection"]
    PAIRS --> A["Action encoder와 projection"]
    S --> MATRIX["Batch 내 state-action similarity matrix"]
    A --> MATRIX
    TARGET["Positive pair 및 false-negative mask"] --> LOSS["InfoNCE 계열 contrastive loss"]
    MATRIX --> LOSS
    LOSS --> OPT["Projection heads 및 선택한 학습 파라미터 업데이트"]
    OPT -.->|학습 시 업데이트| S
    OPT -.->|학습 시 업데이트| A
```

대각선 pair를 positive로 두는 단순 loss에서 시작할 수 있지만, 같은 action이나 동시에 맞는 감정은 negative에서 제외하거나 multi-positive 목표로 처리한다. encoder를 고정한 경우 projection만 업데이트하며 학습 중 가중치가 바뀌는 embedding은 고정 cache로 재사용하지 않는다.

**장점.** candidate embedding을 미리 계산하고 반복 사용할 수 있다. 같은 후보 풀을 사용하는 tool routing·retrieval에 적합하다.

**한계.** state와 후보의 세밀한 교차 attention이 줄어들고, 의미가 비슷한 오답을 구분하는 hard negative가 중요하다. 다른 행의 정답이 현재 행에도 맞으면 in-batch negative가 거짓 negative가 된다. 학습한 temperature가 실제 정답률 보정까지 보장하지는 않는다.

**실습 적용.** 여러 BANKING77 라벨 설명을 cache하는 비교 실험으로 확장할 수 있다. GoEmotions의 동시 감정은 모두 positive로 취급해야 한다. Qwen3.5-0.8B로 바꾸는 것은 backbone 크기와 embedding 품질까지 바꾸는 별도 실험이다.

### 1.8 F — diffusion answer-slot readout

**원리의 배경과 파생 관계.** Austin et al.(2021)의 *D3PM*은 discrete state의 corruption/denoising을 연구한다. [R22](https://arxiv.org/abs/2107.03006) Nie et al.(2025)의 *LLaDA*는 masked token 복원으로 언어 모델을 학습하는 diffusion LM의 사례다(§2). [R23](https://arxiv.org/abs/2502.09992) **typed answer-slot readout의 직접 구현 근거**는 DiffusionGemma를 사용하는 djev 소스다. D3PM/LLaDA는 diffusion의 배경 설명이며, djev 또는 DiffusionGemma가 LLaDA의 직접 파생 모델이라는 의미는 아니다. [R12](https://github.com/mmastrac/djev)

**원리.** discrete diffusion 모델의 answer canvas에 고정 문구를 넣고 답변 slot만 미정 상태로 둔다. 한 denoising step의 slot logits를 읽으면 여러 판단을 병렬로 얻을 수 있다. 여러 step 또는 noise draw를 사용할 수도 있다.

**최근 공개 구현.** djev는 DiffusionGemma에서 typed decision을 읽는 서버를 공개한다. canvas seed, 고정 위치, step 제한, slot logprobs를 사용하며 엔진 변경이 필요한 옵션은 지원 상태를 별도로 확인해야 한다. [R12](https://github.com/mmastrac/djev)

**프로세스 — diffusion answer-slot readout**

```mermaid
flowchart TD
    INPUT["state와 typed questions"] --> CONTEXT["조건 context와 answer schema"]
    CONTEXT --> CANVAS["고정 문구를 pin하고 답변 slot은 미정으로 둔 canvas"]
    NOISE["선택한 noise seed"] --> CANVAS
    CANVAS --> DENOISE["Diffusion backbone의 denoising step"]
    DENOISE --> CHECK{"설정한 step 수에 도달했는가?"}
    CHECK -->|아니오| UPDATE["미정 slot 갱신, 고정 위치 유지"]
    UPDATE --> DENOISE
    CHECK -->|예| READ["각 answer slot의 label logits 또는 logprobs 읽기"]
    READ --> NORMALIZE["후보 집합과 vocab 정규화 기준 확인"]
    NORMALIZE --> OUT["Slot별 Choice, Noul, Score answer"]
```

한 read에서 함께 처리하는 독립 질문의 개념도다. 질문 간 의존성이 있으면 여러 stage가 필요할 수 있다. noise draw를 여러 번 실행하거나 label-only 정규화를 쓰는 옵션은 엔진 지원과 별도 평가가 필요하다. 이 과정 자체가 새 diffusion backbone의 학습을 의미하지는 않는다.

**장점.** 여러 answer 위치를 함께 읽는 실행 형태를 실험할 수 있다.

**한계.** slot 간 의존성, stochastic sampling, vocab 정규화와 step 수를 평가해야 한다. denoising 1 step이 요청 전체의 backbone 연산 1회 또는 확률 보정 완료를 뜻하지 않는다.

**실습 적용.** Qwen3.5-0.8B는 여기서 사용하는 diffusion backbone이 아니므로 본 Qwen 미세조정의 직접 구현 대상에는 넣지 않는다. 향후 다른 backbone과 비교할 때 독립된 실험으로 다룬다.

### 1.9 학습 목표 ① — supervised decision learning

**출처와 이번 적용.** task head의 supervised fine-tuning은 Devlin et al.(2018)의 BERT를 기반 연구로, code readout의 decision SFT는 Jeff 공개 소스를 구현 근거로 인용한다. [R17](https://arxiv.org/abs/1810.04805v2), [R3](https://github.com/firelex/jeff/blob/f06788292874c21a5b5c41549ac220dd9e15da7f/src/jeff/model.py) CE/BCE 자체를 Jev가 제안한 새로운 loss로 보지 않는다. 아래 BANKING77/GoEmotions/SST-5 변환·sampling·split 규칙은 본 노트북의 데이터 설계다.

BANKING77의 하나의 정답 $y$에는 다음 CE가 기본이다.

$$L_{choice}=-\log p_y.$$

독립 명제의 binary 정답 $y\in\{0,1\}$에는 BCE를 적용한다.

$$L_{noul}=-y\log p-(1-y)\log(1-p).$$

GoEmotions를 고정 28-label head로 학습하면 label별 BCE를 평균한다. 감정별 false/true 질문으로 펼치면 같은 문장에서 나온 여러 행을 **같은 split**에 넣고 원본 문장별 가중치를 고려한다. independent sigmoid 확률은 합이 1일 필요가 없다.

세 데이터셋을 섞을 때 단순 row 평균만 쓰면 감정별로 늘어난 GoEmotions가 학습을 지배할 수 있다. 데이터셋별 sampling 또는 loss weight를 기록한다. 표본 balancing으로 양성 비율을 바꾸면 배포 분포에서 확률 보정을 다시 평가한다.

Score는 SST-5의 5단계 감성 극성 정답으로 CE를 학습하고, ordinal loss를 비교 대상으로 둔다. accuracy와 함께 level MAE 및 quadratic weighted kappa(QWK)를 평가한다. 순서가 있다는 이유만으로 class index를 연속 실수 회귀로 바꿀지는 별도 가정이 필요하다. SST-5는 감성 극성 척도이며 GoEmotions 감정별 강도의 정답은 아니다. [SST 원 논문](https://aclanthology.org/D13-1170/)

**학습 프로세스 — supervised decision learning**

```mermaid
flowchart TD
    RAW["BANKING77, GoEmotions, SST-5 원본"] --> SPLIT["원본 ID와 중복을 고려해 split 고정"]
    SPLIT --> TRAIN["Train: state와 question, 정답 변환"]
    TRAIN --> MODEL["선택한 A부터 E 구조의 logits"]
    TRAIN --> TARGET["단일 라벨 또는 multi-hot 정답"]
    MODEL --> CE["Choice, Score 또는 false/true 후보: CE"]
    MODEL --> BCE["독립 명제 또는 multi-label: BCE"]
    TARGET --> CE
    TARGET --> BCE
    CE --> MIX["데이터셋 sampling과 loss weight 적용"]
    BCE --> MIX
    MIX --> OPT["Backward와 optimizer step"]
    OPT -.->|다음 train batch| MODEL
    SPLIT --> VALID["Validation: checkpoint와 하이퍼파라미터 선택"]
    MODEL --> VALID
    VALID --> CAL["별도 calibration fold에서 확률 보정"]
    CAL --> TEST["고정 test에서 최종 평가"]
```

split을 먼저 정한 뒤 감정별 Noul 행으로 펼친다. supervision 표현에 따라 CE/BCE를 고르고, validation·calibration·test의 역할을 구분한다. A부터 E 모두에 동일한 loss를 강제한다는 뜻은 아니며, E는 contrastive loss와 병행할 수도 있다.


### 1.10 학습 목표 ② — teacher distillation과 hard negatives

**원리의 배경과 파생 관계.** Hinton, Vinyals & Dean(2015)의 *Distilling the Knowledge in a Neural Network*, §2는 teacher의 softened distribution을 작은 student에 전달하는 대표적 근거다. [R24](https://arxiv.org/abs/1503.02531v1) 아래 식은 $T=1$인 soft-target CE 예시다. $T>1$의 classic distillation을 사용할 때는 teacher와 student 양쪽 logits에 같은 temperature를 적용하고, hard-label loss 혼합과 distillation 항의 $T^2$ scaling도 별도로 정한다. hard-negative 선택은 contrastive 학습의 negative 구성을 다루는 축이며 soft-target 증류와 같지 않다. [R21](https://arxiv.org/abs/1807.03748v2), [R11](https://github.com/Contrastive-LM/CLM)

더 강한 teacher의 판단을 student에 전달하려면 one-hot 정답 대신 teacher의 후보 분포 $q$를 사용할 수 있다.

$$L_{soft}=-\sum_k q_k\log p_k.$$

같은 후보 분포를 대상으로 한 KL divergence는 teacher가 고정일 때 soft CE와 같은 student 최적화 방향을 갖는다. teacher와 student의 후보 순서·설명·label mapping을 맞추어야 한다.

**적용 방법.** 사람이 검수한 원본 정답을 유지하면서 교사 분포를 별도 필드로 저장한다. “환불 요청”과 “환불이 아직 반영되지 않음”처럼 헷갈리는 후보를 hard negative로 넣는다. teacher의 애매한 분포를 강제로 정답 하나로 바꾸면 불확실성 정보가 줄어든다.

**한계.** teacher의 오류·과신·프롬프트 편향도 전달된다. 특정 데이터셋에 이미 학습된 teacher로 만든 성능 개선을 unseen-task 일반화로 해석하지 않는다. 상용 API teacher 없이도 공개 모델이나 원본 라벨만으로 첫 실습을 진행할 수 있다.

이 절의 distillation 설계는 본 실습의 확장 제안이다. CLM의 hard-negative refinement는 관련 예시이지만 동일한 teacher distillation 방법이라는 뜻은 아니다. [R11](https://github.com/Contrastive-LM/CLM)

**학습 프로세스 ① — teacher distillation**

```mermaid
flowchart TD
    INPUT["같은 state, question, 후보와 code mapping"] --> TEACHER["고정 teacher"]
    INPUT --> STUDENT["학습할 student"]
    TEACHER --> Q["Temperature 적용한 teacher 분포 q"]
    STUDENT --> P["같은 temperature의 student 분포 p"]
    Q --> SOFT["Soft-target CE 또는 KL"]
    P --> SOFT
    STUDENT --> HARD["원본 정답에 대한 supervised loss"]
    LABEL["사람이 검수한 원본 정답"] --> HARD
    SOFT --> MIX["Soft loss와 hard loss 가중 합"]
    HARD --> MIX
    MIX --> OPT["Student 파라미터만 업데이트"]
    OPT -.->|학습 시 업데이트| STUDENT
```

teacher 분포는 gradient를 끊고 사용한다. $T>1$이면 distillation 항의 $T^2$ scaling과 혼합 가중치를 명시한다. 단순 soft-target 예시는 $T=1$이다.

**학습 프로세스 ② — hard-negative 구성**

```mermaid
flowchart TD
    POS["정답 positive action 또는 label"] --> POOL["의미가 비슷한 후보 검색 또는 모델 오답 수집"]
    POOL --> VERIFY["실제로 오답인지 검수: 또 다른 정답은 제외"]
    VERIFY --> GROUP["Positive와 쉬운 negative, hard negative로 후보 묶음 구성"]
    GROUP --> SCORE["Student로 후보 점수 계산"]
    SCORE --> LOSS["선택한 구조의 CE 또는 contrastive loss"]
    LOSS --> OPT["Student 업데이트"]
    OPT -.->|필요할 때 다시 mining| POOL
```

hard-negative mining은 **어떤 오답 후보를 학습에 보여줄지** 정하는 과정이다. teacher 분포를 전달하는 distillation과 별도로 적용할 수 있다. test 결과를 mining에 사용하지 않는다.


### 1.11 학습 목표 ③ — calibrated decision과 RL

**TypeSafe의 RLCD.** TypeSafe는 Reinforcement Learning for Calibrated Decisions라는 학습 접근과 병렬 decision sampling을 소개한다. 그러나 확인한 공식 발표만으로 backbone, readout, reward 식, 데이터와 optimizer를 특정할 수 없다. 다음 논문을 RLCD의 공개 구현으로 간주하지 않는다. [R1](https://typesafe.ai/blog/introducing-system-one-models-and-jev)

**관련 연구 1 — RLCR(2025-07).** *Beyond Binary Rewards*는 correctness reward에 Brier score 기반 보상을 결합해 정확도와 confidence를 함께 다룬다. reasoning과 수치 confidence를 출력하는 모델의 학습 연구이므로 prefill-only decision head에 그대로 옮길 수는 없다. proper scoring rule이 calibration 목적에 왜 유용한지 이해하는 참고이다. [R10](https://arxiv.org/abs/2507.16806v1)

**관련 연구 2 — Calibration Aware RL(2026-01).** Yaldiz 등은 SFT/RLVR의 정확도–calibration tradeoff와 decision-token 과신을 연구하고 decision-token 확률을 직접 조정하는 접근을 제안한다. 결과는 해당 실험 조건에서의 관찰이며 모든 SFT가 잘 보정되거나 모든 RL이 과신한다는 보편 법칙은 아니다. [R9](https://arxiv.org/abs/2601.13284v1)

이 실습은 우선 CE/BCE 학습과 post-hoc calibration을 구축한다. RL을 도입하려면 reward, rollout, label·probability 정의와 비용을 추가로 설계해야 한다. 단순히 Brier loss를 CE에 더한 supervised objective를 “RLCD 재현”이라고 부르지 않는다.

**관련 RL 연구의 개념도 — rollout과 calibration 보상**

```mermaid
flowchart TD
    TASK["정답을 검증할 수 있는 문제 batch"] --> POLICY["현재 policy 모델"]
    POLICY --> ROLLOUT["연구별 rollout: 답과 confidence 또는 decision-token 확률"]
    ROLLOUT --> CHECK["정답과 비교한 correctness"]
    ROLLOUT --> PROB["Confidence 또는 확률의 scoring 항"]
    GOLD["검증 정답"] --> CHECK
    GOLD --> PROB
    CHECK --> REWARD["연구별 correctness와 calibration reward 또는 objective"]
    PROB --> REWARD
    REWARD --> UPDATE["해당 연구의 RL 학습 규칙으로 policy 업데이트"]
    UPDATE -.->|다음 rollout| POLICY
    POLICY --> EVAL["분리된 평가 데이터: 정확도와 calibration"]
```

RLCR [R10](https://arxiv.org/abs/2507.16806v1)과 Calibration Aware RL [R9](https://arxiv.org/abs/2601.13284v1)을 읽기 위한 공통 단계의 개념도다. 두 연구의 rollout·reward·objective·optimizer가 같다는 뜻은 아니다. **TypeSafe RLCD의 비공개 내부 알고리즘을 표현한 그림이 아니다.**


### 1.11.1 Laya와 decider: calibration을 어디서 만드는가

**분석일: 2026-10-03.** Laya 소스 `fa9a2a7`, decider 소스 `4502408`을 고정해 읽었다. TypeSafe RLCD의 정확한 reward·optimizer는 공개 자료에서 확인되지 않아, 두 프로젝트는 **공개된 학습 방향을 비교하는 독립 구현**으로 다룬다.

Calibration은 “0.8이라고 예측한 사건들이 반복 관측에서 약 80% 발생하는가”의 문제다. 이를 세 층으로 나눈다.

| 층 | Laya 공개 fine-tuning | decider-2B v8 → v10 |
| --- | --- | --- |
| 확률을 읽는 구조 | ModernBERT/mmBERT → 후보 marker 표현 → shared scalar scorer | Qwen3.5 → answer slot hidden state → 선택지 letter logits |
| 학습 중 확률 목표 | teacher soft distribution에 proper-score reward + soft CE | 실제 행동 outcome의 PPO + 다음 outcome 확률의 log score |
| 학습 후 보정 | 고정 logits의 NLL을 최소화하는 temperature | RL 단계는 기존 T=1.30 유지; 별도 도구가 type별 T를 fitting |
| 보정 후 자동 처리 정책 | max(p)의 bucket별 threshold, 선택적 histogram binning | threshold/abstention 설정과 별도 평가 |

**읽은 범위.** Laya는 공개 MPS fine-tuning 코드의 recipe를 분석했다. 이것이 배포된 모든 Laya checkpoint의 최초 학습 이력을 완전히 재현한다는 뜻은 아니다. decider는 공개 RL 문서와 inference/calibration 코드를 읽었으며, RL trainer는 별도 research 저장소에 있다고 문서에 명시돼 있다. decider 전체 모델에 RL 이력을 일괄 적용하지 않는다. 2B v10의 RL, v11의 추가 supervised LoRA, 4B 모델 카드의 RL 없는 경로를 구분한다.

출처: [Laya model/reward](https://github.com/NandhaKishorM/laya/blob/fa9a2a7070b1789912a49ae24603bbfb1a78b001/laya/common.py), [Laya MPS 학습](https://github.com/NandhaKishorM/laya/blob/fa9a2a7070b1789912a49ae24603bbfb1a78b001/notebooks/laya_finetune_typed_decisions_mps.py), [decider readout](https://github.com/Mapika/decider/blob/45024082b7d4bb667bf9140b7a7c073d3b483f6b/decider/model.py), [decider RL recipe](https://github.com/Mapika/decider/blob/45024082b7d4bb667bf9140b7a7c073d3b483f6b/docs/RL.md), [2B card](https://github.com/Mapika/decider/blob/45024082b7d4bb667bf9140b7a7c073d3b483f6b/MODEL_CARD.md), [4B card](https://github.com/Mapika/decider/blob/45024082b7d4bb667bf9140b7a7c073d3b483f6b/MODEL_CARD_4B.md).

### 1.11.2 Laya 학습: 확률 분포 자체에 보상을 주기

공개 fine-tuning의 target $t$는 `gold[question_id].probabilities`를 후보 순서대로 모아 정규화한 **teacher 분포**다. 관측된 실제 사건의 빈도와 teacher 확률은 따로 평가해야 한다.

`proper_reward(q, target, ...)`는 다음 항들을 계산한다. $q$는 모델이 보고한 확률, $t$는 target 분포다.

$$R(q,t)=\sum_k t_k\log q_k+0.75\frac{\sum_k t_kq_k}{\sqrt{\sum_kq_k^2}}-\mathbf{1}_{\mathrm{Score}}\,RPS(q,t).$$

$$RPS(q,t)=\frac{1}{K-1}\sum_{k=1}^{K}\left(\sum_{j\leq k}q_j-\sum_{j\leq k}t_j\right)^2.$$

- **Log score:** 실제/target 확률이 있는 후보에 거의 0을 주면 큰 손해를 받는다.
- **Spherical score:** target 분포와의 정렬을 보상한다.
- **RPS:** 순서형 Score에서 누적분포의 차이를 벌점으로 준다. 먼 level에 확률을 잘못 두는 오류를 구분한다.

함수의 기본 spherical 가중치는 0.5지만 공개 MPS/Kaggle fine-tuning 호출은 **0.75**를 사용한다. 구현은 log 값을 **−9.21에서 clipping**한다. 원형 scoring rule의 proper 성질과 이 clipping을 포함한 모든 수치 영역의 성질을 동일하게 단정하지 않는다.

**업데이트 절차.** 한 입력의 logits에 평균을 제거한 Gaussian noise를 네 번 더해 네 분포를 만든다. 각 분포의 reward에서 네 표본의 평균을 빼고 표준편차로 정규화해 advantage를 만든다. noise는 epoch에 따라 σ=0.4 → 0.1로 줄인다.

$$L_{\mathrm{RL}}=-\operatorname{mean}[A\log P_\theta(\tilde z)],\qquad L_{\mathrm{CE}}=-\sum_k t_k\log\operatorname{softmax}(z)_k.$$

여기서 policy log-probability는 **샘플링한 noisy logits의 Gaussian density**에 대응한다. autoregressive 문장을 생성하거나 환경에서 행동을 실행하는 rollout은 이 fine-tuning 루프에 없다. 최종 loss는 `loss_rl + loss_ce`; encoder와 decision scorer를 함께 업데이트한다. 코드의 `0.0 * activation.sum()`은 act head에 보상을 학습시키는 항이 아니다.

**해석.** 이 경로는 teacher의 확률을 잘 맞추도록 설계된 distribution learning이다. 실제 outcome calibration을 주장하려면 teacher와 독립된 실제 정답/결과로 검증해야 한다. CE도 proper log-score loss이므로, RL 항의 추가 효과는 `CE만`과 `CE+RL`을 같은 데이터·평가에서 비교해야 알 수 있다.

출처: [target 구성과 학습 루프](https://github.com/NandhaKishorM/laya/blob/fa9a2a7070b1789912a49ae24603bbfb1a78b001/notebooks/laya_finetune_typed_decisions_mps.py#L58), [proper_reward](https://github.com/NandhaKishorM/laya/blob/fa9a2a7070b1789912a49ae24603bbfb1a78b001/laya/common.py#L604).

### 1.11.3 Laya 학습 후 보정: type별 T와 후보 수별 T

**MPS fine-tuning의 마지막 단계.** 학습 전에 tokenized item 중 `min(calib_max, 전체 item 수 // 10)`을 고정 seed로 떼어 두고, 최종 모델의 logits를 모은다. Choice·Score·Noul 각각에 대해 **log T를 LBFGS로 최적화**한다.

$$T^*=\arg\min_{T>0}-\frac{1}{N}\sum_{i,k}t_{ik}\log\operatorname{softmax}(z_i/T)_k.$$

모델 가중치는 고정된다. 여기서도 $t$는 teacher soft target이다. MPS script의 fit은 T를 `[0.1, 10]`으로 제한하고, type에 표본이 10개 미만이면 1.0을 반환한다. 결과는 `rl_agent_config.json`의 `temperature` 배열 **[Choice, Score, Noul]**로 저장한다. 기존 `temperature_by_options`는 제거해 예전 bucket 값이 새 type별 보정을 덮어쓰지 않게 한다.

**현재 라이브러리의 더 세분화된 보정.** `laya/calibrate.py`는 type × 후보 수 bucket(`2`, `3–5`, `6–10`, `11+`)별 T도 지원한다. bucket fit에는 최소 2,000개, type fallback에는 최소 10개를 요구한다. 이 라이브러리와 checkpoint load의 T 범위는 **[0.5, 5.0]**이다. fine-tuning script의 [0.1,10]과 다르므로 저장한 값과 실제 적용 값을 함께 기록해야 한다.

`fit_temperature_map(..., compute_ece=True)`는 bucket별 20%를 평가용으로 제외한다. 남은 fit 표본이 2,000개보다 적어지는 bucket은 전부 fit에 쓰고 **ECE 평가에서 제외**하며, 제외 목록을 반환한다. 기본 `compute_ece=False`는 입력 전체를 fit에 쓰고 평가 결과는 만들지 않는다.

라이브러리의 ECE report는 soft target을 `argmax(target)`으로 hard label화해 평가한다. teacher soft target을 입력했다면 이 ECE는 teacher의 최상위 선택과의 일치에 대한 수치이며, 실제 outcome의 확률 calibration을 직접 입증하지 않는다.

**confidence 필드 확인.** `answer_confidence`는 모든 primitive에서 `max(p)`다. Choice·Score의 기존 `confidence`는 $1-H(p)/\log K$이며 분포 집중도다. Noul의 `confidence`는 max(p)다. Noul의 명제 확률 `noul=P(true)`와 선택된 답의 정답 확률 `max(P(false),P(true))`도 구분한다. ECE/자동 처리에는 측정하려는 사건과 같은 값을 써야 한다.

라이브러리의 `fit_binning_map`은 confidence 구간을 실제 정답률로 매핑하는 **top-label confidence 보정 도구**다. 자동으로 K개 후보 분포 전체를 다시 학습하는 기능은 아니다. `fit_abstention_thresholds`는 bucket별 허용 오류율에 맞춰 처리 임계값을 고르는 정책 도구이며, 자체로 calibration을 보장하지 않는다.

**분할 관찰.** 공개 fine-tuning 코드는 질문으로 펼친 item을 무작위 분할한다. 같은 state의 다른 질문이 train과 calibration 양쪽에 들어갈 수 있다. 우리 실습에서는 먼저 원본 case/document/task ID로 분할하고 그 안에서 질문을 펼친다.

출처: [MPS split·fitter·저장](https://github.com/NandhaKishorM/laya/blob/fa9a2a7070b1789912a49ae24603bbfb1a78b001/notebooks/laya_finetune_typed_decisions_mps.py#L198), [현재 라이브러리 fitter](https://github.com/NandhaKishorM/laya/blob/fa9a2a7070b1789912a49ae24603bbfb1a78b001/laya/calibrate.py), [runtime decode](https://github.com/NandhaKishorM/laya/blob/fa9a2a7070b1789912a49ae24603bbfb1a78b001/laya/agent.py#L1188), [temperature 범위/confidence 정의](https://github.com/NandhaKishorM/laya/blob/fa9a2a7070b1789912a49ae24603bbfb1a78b001/laya/common.py#L664).

### 1.11.4 decider-2B v10: 행동 성공과 outcome belief를 함께 학습

공개 RL 문서의 대상은 **v8 → v10**이다. 다음 내용은 저자의 recipe와 보고 결과이며, 이 저장소에서 원래 RL trainer를 실행해 재현한 결과가 아니다.

- **Actor, 계수 0.1:** 모델 확률에서 실제 행동을 샘플링하고 환경의 성공/실패로 PPO clipped loss(clip 0.2)를 계산한다. 같은 환경 네 번 실행의 leave-one-replicate-out baseline을 쓴다.
- **Belief, 계수 0.2:** 다음 행동의 결과를 확률로 예측하고 log score를 적용한다. 미끄러운 grid의 전이 확률, bag 구성, minesweeper posterior는 환경 법칙을 쓴다. 브라우저는 실제 click outcome을 쓴다.
- **Rendering consistency, 계수 1.0:** state-first/schema-first와 원래/역순 후보 표현의 분포를 KL로 맞춘다. 문자열 배치와 후보 순서가 바뀌어도 확률이 크게 흔들리지 않도록 한다.
- **Retention, 계수 0.7:** replay row에서 KL(v8 ‖ student)로 기존 능력을 유지한다. 평균 KL>0.01 nats 또는 어떤 row의 KL>0.05면 actor/belief를 중지하고 문서상 retention 항만 적용한다.

```mermaid
flowchart LR
    S["환경 state"] --> D["decider가 행동 확률과 outcome belief 출력"]
    D --> A["행동 샘플링·실행"]
    A --> O["실제 성공·실패"]
    O --> PPO["Actor PPO"]
    LAW["알려진 확률 법칙 또는 관측 outcome"] --> LOG["Belief log-score"]
    D --> LOG
    PPO --> U["가중치 업데이트 + consistency/retention"]
    LOG --> U
    U -.-> D
```

RL 단계에서는 **served distribution의 T=1.30을 그대로 유지**했다. 따라서 이 단계의 개선은 새로운 temperature를 fitting해서 얻은 변화로 설명되지 않는다. 저자는 exact-law 대비 belief log-score gap이 **0.473 → 0.219 nats**로 줄었다고 보고한다. 일반 Bespoke suite의 calibration은 이 단계에서 변하지 않았다고 함께 명시한다. outcome belief의 개선을 모든 질문의 ECE 개선으로 확대하지 않는다.

**Laya와의 차이.** Laya 공개 fine-tuning의 reward target은 teacher distribution이고, decider v10의 belief는 실제 환경의 법칙/관측에 연결된다. 행동 성공률을 올리는 actor 항과 확률을 정직하게 만드는 belief 항을 분리해 볼 수 있다는 점이 분석에 유용하다. TypeSafe RLCD와 동일한 알고리즘이라는 증거는 없다.

출처: [v10 RL 문서](https://github.com/Mapika/decider/blob/45024082b7d4bb667bf9140b7a7c073d3b483f6b/docs/RL.md), [2B model card](https://github.com/Mapika/decider/blob/45024082b7d4bb667bf9140b7a7c073d3b483f6b/MODEL_CARD.md).

### 1.11.5 decider 학습 후 보정: 실제 serving readout을 그대로 fitting

`decider/calibrate.py`는 **T=1로 수집한 logits 또는 probabilities와 gold index**를 입력받는다. type별 정답 NLL을 최소화하는 T를 `[0.05, 20]`의 801점 log grid에서 찾고, 인접 구간을 golden-section search 40회로 세분화한다. `fit_by_type`의 기본 최소 표본은 type당 50개다. 표본이 부족한 type은 map에서 빠지고 config의 공통 T를 사용한다.

이 함수는 **split을 자동으로 만들지 않는다.** 출력 `nll`의 전후 비교는 fitting에 쓴 같은 records의 NLL이다. 이를 test calibration 향상이라고 보고하지 않고, caller가 별도 test에서 NLL·Brier·ECE를 계산해야 한다.

**Score의 중요한 구현 차이.** isolated-levels 경로는 level별 `[no, yes]` 두 logits를 먼저 보정하고 다음처럼 합친다.

$$r_k(T)=\operatorname{softmax}([z_{k,no},z_{k,yes}]/T)_{yes},\quad p_k(T)=\frac{r_k(T)}{\sum_jr_j(T)}.$$

fitter는 이 **최종 Score 분포**의 gold NLL을 최소화한다. 우리가 단순 5-way softmax Score로 구현한다면 이 경로와 같은 보정이라고 부르지 않는다. 가중치·readout·candidate 순서·모델 정밀도·schema layout을 고정하고 그 경로에서 logits를 수집해야 한다.

type별 `temperature_by_type`, schema-first용 별도 map, 일부 stock `decider-chat` 모델의 후보 수 함수 $T(n)=\max(T_{min},a+b\log n)$도 runtime에 존재한다. 후보 수 함수는 `fit_by_type`가 자동으로 학습하는 값이 아니다. scalar override가 type map을 끄는 경우도 있어 저장한 config와 실제 적용 값을 같이 확인한다.

출처: [공개 calibration fitter](https://github.com/Mapika/decider/blob/45024082b7d4bb667bf9140b7a7c073d3b483f6b/decider/calibrate.py), [runtime temperature 규칙](https://github.com/Mapika/decider/blob/45024082b7d4bb667bf9140b7a7c073d3b483f6b/decider/temperature.py), [readout](https://github.com/Mapika/decider/blob/45024082b7d4bb667bf9140b7a7c073d3b483f6b/decider/model.py).

### 1.11.6 이 컴퓨터에서 확인한 범위

고정한 두 소스의 **실제 calibration 함수**를 모델 다운로드 없이 CPU에서 실행했다. 합성 입력의 true probability는 {0.2,0.4,0.6,0.8} 중 하나이고, logits를 의도적으로 4배 과신하게 만들었다. NumPy seed 17로 calibration 4,000개와 독립 test 2,000개의 outcome을 생성했다. Laya는 calibration 내부의 800개를 추가로 제외하고 fitting했으며 decider는 4,000개를 사용했다. T 차이는 fitting 표본 차이를 포함한다.

| 합성 실험 | 적용 T | Test accuracy | 평균 max(p) | Test NLL | Test ECE(15 bins) |
| --- | --- | --- | --- | --- | --- |
| 원래 과신 logits | 1.000 | 0.6995 | 0.9150 | 0.9587 | 0.2155 |
| Laya 공개 fitter | 4.014 | 0.6995 | 0.6987 | 0.5845 | 0.0056 |
| decider 공개 fitter | 4.122 | 0.6995 | 0.6945 | 0.5846 | 0.0072 |

이 값들은 **Laya/decider 모델 성능이 아니라 fitter의 동작 확인**이다. 두 모델의 가중치·RL rollout·원래 training run은 실행하지 않았다. 같은 logits에 양의 T를 적용해 확률의 과신이 줄었고 argmax accuracy는 그대로였다.

Laya의 실제 `proper_reward`에도 target=[0.7,0.3]을 넣어 확인했다. q=[0.7,0.3]은 −0.0397, 과신 q=[0.99,0.01]은 −0.8613, q=[0.5,0.5]는 −0.1628이었다(spherical=0.75, Choice). 이 fixture에서는 실제 분포를 보고한 쪽이 높은 보상을 받는다.

아래 셀은 소스 clone이나 모델 설치 없이 같은 보정 원리를 관찰하는 **독립 교육용 구현**이다. 위 두 공개 fitter의 동일 구현은 아니다.

In [1]:
# 합성 probability/outcome만 사용한다. 모델 학습·다운로드를 하지 않는다.
import math
import random

rng_calibration = random.Random(17)
rng_evaluation = random.Random(23)

def calibration_demo_rows(n, rng):
    rows = []
    for _ in range(n):
        truth = rng.choice((0.2, 0.4, 0.6, 0.8))
        gold = int(rng.random() < truth)
        logit = 4.0 * math.log(truth / (1.0 - truth))
        rows.append((logit, gold))
    return rows

def calibration_demo_nll(rows, temperature):
    # Binary NLL = softplus(z/T) - y*z/T; 수치적으로 안정적인 식이다.
    return sum(max(z / temperature, 0.0) + math.log1p(math.exp(-abs(z / temperature)))
               - y * z / temperature for z, y in rows) / len(rows)

def calibration_demo_metrics(rows, temperature):
    ps = [1.0 / (1.0 + math.exp(-z / temperature)) for z, _ in rows]
    correct = [int((p >= 0.5) == bool(y)) for p, (_, y) in zip(ps, rows)]
    confidence = [max(p, 1.0 - p) for p in ps]
    ece = 0.0
    for bucket in range(15):
        ids = [i for i, c in enumerate(confidence) if min(int(c * 15), 14) == bucket]
        if ids:
            ece += len(ids) / len(rows) * abs(sum(confidence[i] for i in ids) / len(ids)
                                            - sum(correct[i] for i in ids) / len(ids))
    return {"accuracy": round(sum(correct) / len(rows), 4),
            "mean_max_p": round(sum(confidence) / len(rows), 4),
            "nll": round(calibration_demo_nll(rows, temperature), 4),
            "ece_15_bins": round(ece, 4)}

fit_rows = calibration_demo_rows(2000, rng_calibration)
evaluation_rows = calibration_demo_rows(2000, rng_evaluation)
temperature_grid = [math.exp(math.log(0.5) + i / 300 * math.log(10)) for i in range(301)]
fitted_t = min(temperature_grid, key=lambda t: calibration_demo_nll(fit_rows, t))
print("synthetic only; fitted T:", round(fitted_t, 3))
print("before:", calibration_demo_metrics(evaluation_rows, 1.0))
print("after: ", calibration_demo_metrics(evaluation_rows, fitted_t))

synthetic only; fitted T: 4.002
before: {'accuracy': 0.706, 'mean_max_p': 0.9167, 'nll': 0.9698, 'ece_15_bins': 0.2107}
after:  {'accuracy': 0.706, 'mean_max_p': 0.7013, 'nll': 0.5847, 'ece_15_bins': 0.0068}


### 1.11.7 우리 실습에 반영할 순서

1. 원본 case/document/task 단위로 **train / checkpoint validation / calibration / test**를 먼저 나눈다. 여러 질문과 후보 순서 변형은 같은 split에 둔다.
2. 같은 readout에서 CE/BCE baseline을 만들고 원본 logits를 저장한다. hard label·teacher soft label·환경의 true probability를 구분한다.
3. 모델을 고정해 type별 NLL temperature를 fitting하고 독립 test에서 NLL·Brier·ECE를 비교한다. Noul은 P(true)의 event calibration도 별도로 본다. Score는 전체 level 분포와 ordinal 지표를 함께 본다.
4. Laya식 teacher-distribution 실험은 **CE만 / CE+noisy-logit RL**을 같은 seed·data budget으로 비교한다. log-score 효과와 RL update 효과를 분리한다.
5. 실제 outcome을 학습하려면 알려진 확률 법칙이 있는 작은 stochastic 환경에서 decider식 actor/belief를 분리해 비교한다. belief NLL과 행동 성공률을 각각 보고한다. 공개 문서만으로 original decider RL run을 재현했다고 주장하지 않는다.
6. 보정과 별도로 threshold를 선택하고 후보 수·domain·language별 risk/coverage를 확인한다. 모델/LoRA/양자화/readout이 바뀌면 재보정한다.

**이번 선택.** 이 Mac에서 calibration 메커니즘을 구현하기에는 Laya의 공개 MPS recipe가 출발점으로 구체적이다. 실제 outcome에 근거한 calibrated decision을 설계할 때는 decider v10의 actor/belief 분리가 더 직접적인 참고다. 둘의 우열은 동일한 데이터와 평가로 비교하기 전에는 정하지 않는다.

### 1.12 확률 보정과 의사결정 정책

**Temperature scaling.** 학습이 끝난 logits $z$에 양의 scalar $T$를 적용하고 독립 calibration split의 NLL을 최소화한다. [R13](https://proceedings.mlr.press/v70/guo17a.html)

$$p=\operatorname{softmax}(z/T),\qquad
T^*=\arg\min_{T>0} -\sum_i\log p_{i,y_i}(T).$$

양의 scalar temperature는 후보 순위를 유지하면서 분포의 날카로움을 바꾼다. Noul은 logit에 sigmoid($z/T$)를 적용한다. 서로 다른 태스크나 후보 수에 하나의 $T$를 쓸지는 별도 비교 실험이다. calibration split에서 개선되어도 다른 데이터 분포에서 유지된다고 보장되지 않는다.

**평가 지표.** Choice의 accuracy와 macro-F1, Noul/multi-label의 precision·recall·F1에 더해 NLL·Brier·ECE를 함께 본다. ECE는 bin 정의와 표본 수에 민감하므로 reliability diagram과 태스크별 결과도 기록한다.

**정책 평가.** $\max p\geq t$이면 자동 처리하는 경우 threshold $t$별 coverage(자동 처리 비율)와 risk(처리된 항목의 오류율)를 측정한다. 충분한 calibration이 있어도 사건별 비용이 다르면 최적 행동이 달라진다. abstain 또는 `none of the above`는 label 정의와 검수 예제를 따로 마련한다.

**프로세스 — post-hoc temperature와 처리 정책**

```mermaid
flowchart TD
    CKPT["학습이 끝난 고정 checkpoint"] --> CALLOG["Calibration fold의 logits"]
    CALLOG --> FIT["정답 NLL을 최소화하는 양의 T 선택"]
    GOLD["Calibration fold 정답"] --> FIT
    FIT --> T["고정한 temperature T"]
    NEW["새 입력 또는 최종 test 입력"] --> MODEL["고정 모델의 logits z"]
    CKPT --> MODEL
    MODEL --> SCALE["z를 T로 나누기"]
    T --> SCALE
    SCALE --> P["Choice는 softmax, 독립 Noul은 sigmoid"]
    P --> POLICY{"사전에 정한 처리 기준 충족?"}
    POLICY -->|예| AUTO["자동 처리"]
    POLICY -->|아니오| REVIEW["사람 검토 또는 abstain"]
    TH["Validation에서 선택한 정책 threshold와 비용 규칙"] --> POLICY
    P --> EVAL["Test: NLL, Brier, ECE 및 risk-coverage"]
```

temperature는 calibration fold에서, 정책 threshold는 test를 제외한 validation 자료에서 정한다. 그림의 Noul은 단일-logit sigmoid 경로이며 false/true 두 후보 경로는 softmax의 true 확률을 쓴다. 모델 가중치는 이 보정 단계에서 업데이트하지 않는다.


### 1.13 2026년 9월 연구가 추가한 평가 질문

**확률 일관성 — Do System One Decisions Add Up?(2026-09-27).** 이 preprint는 Jev와 Laya를 대상으로 직접 세부 라벨 분포를 구했을 때와 상위 라벨→세부 라벨 경로로 재구성했을 때의 불일치를 연구한다. 개별 softmax 합이 1이어도 계층형 workflow에서 분포가 일관되지는 않을 수 있다. [R14](https://arxiv.org/abs/2609.33971v1)

→ BANKING77에서 `카드/송금/계정` 같은 상위 분류를 도입한다면 직접 분류와 단계 분류의 확률을 비교한다. 어느 경로가 더 정확한지도 따로 측정한다.

**적대적 입력 — JevAdvBench(2026-09-25).** 이 preprint는 state나 질문의 변경에 따른 typed decision의 변화를 연구한다. 같은 요청의 재실행 baseline과 공격 입력 변화의 효과를 구분하는 설계가 중요하다. clean response와 달라졌다는 사실만으로 오류가 증가했다고 단정할 수는 없다. [R15](https://arxiv.org/abs/2609.31142v1)

→ 정답이 있는 test에서 관련 없는 문구 추가, 의견 삽입, 후보 순서 변경, 질문 paraphrase에 대한 accuracy와 확률 변화를 기록한다. 데이터 문장을 명령으로 따르는지도 점검한다.

두 논문은 **평가 연구**이며 새로운 backbone 학습 방법의 근거가 아니다. 최신 preprint의 결과를 독립 재현된 제품 성능으로 표현하지 않는다.

### 1.14 Qwen3.5-0.8B: 환경 선택과 공통 구현 순서

Qwen3.5-0.8B는 Qwen 공식 모델 카드의 구조·tokenizer·chat template를 기준으로 사용한다. text-only decision 실습은 text backbone을 사용한다. Transformers에는 PyTorch `Qwen3_5ForCausalLM` 구현이 있고, MLX-LM에도 Qwen3.5 구현이 있다. 이것은 backend 구현의 존재를 확인한 것이며 아래 decision trainer의 학습 성공을 검증한 것은 아니다. [R2](https://huggingface.co/Qwen/Qwen3.5-0.8B), [R27](https://huggingface.co/docs/transformers/model_doc/qwen3_5), [R16](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md)

**각자의 환경에서 시작하기.** 설치 명령은 OS·driver에 맞는 공식 selector를 사용하고, 실습 환경의 Python·라이브러리 버전을 checkpoint metadata에 기록한다. 아래는 앞으로 구현할 경로와 최초 점검 범위다. [R26](https://pytorch.org/get-started/locally/)

| 환경 | 실습 경로 | 먼저 할 실습 | 확인할 제약 |
| --- | --- | --- | --- |
| Linux / Windows / macOS, GPU 없음 | PyTorch + Transformers, CPU | 소량 입력의 logits·loss 확인 → frozen backbone + readout | CPU float32부터 시작. backbone을 고정해도 forward 비용과 가중치 메모리는 필요. 속도를 본 뒤 범위를 확대 |
| NVIDIA GPU 로컬 / 원격 / notebook 서비스 | PyTorch + Transformers, CUDA | readout-only → LoRA+readout | driver와 PyTorch wheel의 호환, dtype·attention/kernel 지원 확인. CUDA 전용 최적화에 의존하는 코드는 CPU 경로와 구분 |
| Apple Silicon Mac | MLX + MLX-LM | 같은 데이터로 readout-only → custom decision loss + LoRA | MLX module·gradient·adapter 저장·재로드를 별도 구현. Jeff MLX backend는 추론 참고 구현 |
| Mac에서 PyTorch 사용 | PyTorch CPU; MPS는 추가 선택 | CPU 기준 결과 확인 후 MPS forward/backward 점검 | MPS 사용 가능 여부만으로 Qwen3.5의 모든 연산 지원을 단정하지 않음. 오류 시 CPU 경로 사용 |
| 지원되는 AMD GPU / Linux | PyTorch ROCm, 선택적 확장 | CPU 결과와 작은 batch 비교 | 지원 GPU·ROCm wheel·필요 연산 확인. PyTorch ROCm은 `torch.cuda` API를 사용하므로 NVIDIA CUDA와 metadata에서 구분 |

PyTorch의 CPU/CUDA/ROCm 설치 안내와 MPS backend 문서를 근거로 환경을 나눈다. 구체적 decision 학습의 호환성은 3번에서 작은 forward/backward로 확인한다. [R26](https://pytorch.org/get-started/locally/)

**MLX 범위.** 현재 공식 MLX 설치 문서는 Apple Silicon뿐 아니라 Linux CUDA와 CPU-only 패키지도 설명한다. 따라서 MLX를 “Mac 전용”으로 일반화하지 않는다. 이 노트북의 기본 MLX 실습 대상은 Apple Silicon이며, Linux MLX는 MLX-LM/Qwen3.5 및 custom decision loss의 연산 지원을 검증한 뒤 확장한다. [R28](https://ml-explore.github.io/mlx/build/html/install.html)

**공통 계약.** backend가 달라도 원본 split·state/question·candidate 순서·code mapping·loss·calibration fold·지표를 공유한다. 프레임워크별 adapter나 양자화 checkpoint를 서로 그대로 읽을 수 있다고 가정하지 않는다. 먼저 같은 비양자화 모델의 logits·확률을 비교한다.

| 단계 | 구현 | 학습 파라미터 | 검증 질문 |
| --- | --- | --- | --- |
| 0 | 미세조정 전 code-token 점수 | 없음 | 초기 정확도·과신·순서 편향은? |
| 1 | B 방식의 code readout 초기화 | readout만 | 동일 batch에서 loss와 gradient가 유한한가? |
| 2 | B 방식 + LoRA [R25](https://arxiv.org/abs/2106.09685) | adapter + readout | test 성능과 calibration이 개선되는가? |
| 3 | C 방식 비교 | shared scalar head + adapter | 후보별 연산 비용이 정확도 차이를 설명하는가? |
| 4 | temperature와 policy 평가 [R13](https://proceedings.mlr.press/v70/guo17a.html) | $T$·threshold | 허용 오류율에서 coverage는? |
| 5 | 8-bit / 4-bit 배포 비교 | quantized backbone | 결정·분포·정책이 얼마나 달라지는가? |

**실습 규모.** CPU에서도 데이터 변환·평가 코드와 작은 readout 실험에 참여할 수 있게 먼저 2~4개 라벨, 소량의 train/validation, 짧은 입력, batch 1로 시작한다. 이 값은 성능 평가 기준이 아니라 연산·학습 흐름을 확인할 첫 범위다. 메모리와 시간을 확인한 뒤 라벨·길이·표본을 확대한다. CPU에서는 frozen backbone의 표현을 한 번 저장하고 작은 head만 학습하는 경로도 구성할 수 있다. 원본 split을 바꾸거나 test를 calibration에 사용하지 않는다.

**학습기 구현.** PyTorch는 backbone hidden state와 readout을 `nn.Module`로 구성하고, MLX는 대응하는 MLX module과 gradient 경로를 구현한다. 양쪽 모두 마지막 **유효** 입력 위치와 candidate mask를 처리해 CE/BCE를 계산한다. 일반 next-token 학습 명령만으로 이 decision loss를 구현한 것으로 간주하지 않는다. LoRA는 Hu et al.(2021)을 근거로 사용한다. [R27](https://huggingface.co/docs/transformers/model_doc/qwen3_5), [R16](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md), [R25](https://arxiv.org/abs/2106.09685)

**데이터 목표와 일반화 목표.** BANKING77·GoEmotions·SST-5 세 개만으로 일반적 tool 선택이나 게임 의사결정 능력을 입증하기 어렵다. 먼저 좁은 범위의 태스크 적응을 확인하고 보지 못한 label·question·domain을 별도 평가한다. pretrained Jeff를 추가 학습하는 경우 기존 학습 데이터와 평가 데이터의 중복 가능성도 확인한다.

**공통 실습 프로세스 — 환경 선택부터 배포 비교까지**

```mermaid
flowchart TD
    ENV["OS, 장비와 설치된 패키지 확인"] --> BACKEND{"사용할 backend 선택"}
    BACKEND --> TORCH["PyTorch: CPU 또는 지원 GPU"]
    BACKEND --> MLX["MLX: 선택한 지원 device"]
    TORCH --> CONTRACT["공통 split, tokenizer, code mapping과 평가 계약"]
    MLX --> CONTRACT
    CONTRACT --> BASE["미세조정 전 code-token baseline"]
    BASE --> SMALL["작은 forward와 backward로 연산 지원 확인"]
    SMALL --> HEAD["Readout-only 학습"]
    HEAD --> LORA["환경과 메모리에 맞춰 LoRA와 readout 학습"]
    LORA --> SELECT["Validation checkpoint 선택"]
    SELECT --> CAL["Calibration과 정책 설정"]
    CAL --> RELOAD["Backend에 맞는 checkpoint 재로드"]
    RELOAD --> TEST["고정 test 평가"]
    TEST --> QUANT["지원되는 8-bit, 4-bit 경량화와 재평가"]
    QUANT --> API["공통 typed decision API"]
```

CPU의 작은 readout 실험부터 같은 계약을 사용하고 연산·메모리를 확인한 뒤 학습 범위를 확대한다. 경량화 backend가 지원되지 않는 환경은 비양자화 checkpoint로 API 실습을 진행한다. 3~5번의 상세 구현을 위한 계획도다.


#### 환경 선택 예제 — 패키지 현황만 확인

아래 셀은 직접 실행할 때 OS와 설치된 패키지를 표시하고 실습 경로를 고른다. 자동 선택은 Apple Silicon이면 MLX, 그 외에는 PyTorch다. `BACKEND`를 `"torch"` 또는 `"mlx"`로 바꿀 수 있다. GPU 가용성·연산 호환성·메모리는 이 셀에서 검증하지 않으며 3번의 작은 forward/backward에서 확인한다. 모델 다운로드나 학습은 수행하지 않는다.


In [ ]:
import platform
from importlib import metadata

BACKEND = "auto"  # "auto", "torch", "mlx"
DEVICE = "auto"   # torch: "cpu", "cuda", "mps"; mlx: 별도 stream/device 설정

system, machine = platform.system(), platform.machine().lower()
apple_silicon = system == "Darwin" and machine in {"arm64", "aarch64"}
if BACKEND not in {"auto", "torch", "mlx"}:
    raise ValueError("BACKEND는 auto, torch, mlx 중 하나여야 합니다.")
backend = ("mlx" if apple_silicon else "torch") if BACKEND == "auto" else BACKEND
required = ("torch", "transformers", "peft") if backend == "torch" else ("mlx", "mlx_lm")

print({"os": system, "machine": machine, "backend": backend, "requested_device": DEVICE})
for module in required:
    distribution = module.replace("_", "-")
    try:
        version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        version = "미설치"
    print(f"{distribution}: {version}")
print("다음 단계: 환경별 설치 안내와 작은 forward/backward로 device 지원을 확인하세요.")


### 1.15 References — 원문과 자료의 성격

본문의 `R#` 인용 링크는 아래 항목의 원문으로 연결된다. 각 후보에서 기반 아이디어, 저자가 명시한 직접 계보, 본 실습의 응용을 구분했다. 논문은 arXiv 또는 출판 원문을 링크하며 venue를 확인하지 않은 경우 preprint로 표기한다. 성능 숫자는 본 실습에서 재현하지 않아 방법론 설명에 그대로 전용하지 않았다.

| ID | 자료 / 최초 발표 시점 | 성격 | 이 노트북에서 참고한 부분 |
| --- | --- | --- | --- |
| R1 | TypeSafe, [Introducing System One Models & Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev), 2026-09 | 공식 제품 발표 | typed contract, RLCD 공개 범위 |
| R2 | Qwen Team, [Qwen3.5-0.8B model card](https://huggingface.co/Qwen/Qwen3.5-0.8B) | 공식 모델 문서 | 실제 사용할 backbone·template |
| R3 | firelex, [Jeff](https://github.com/firelex/jeff), [DecisionModel](https://github.com/firelex/jeff/blob/f06788292874c21a5b5c41549ac220dd9e15da7f/src/jeff/model.py), [MLX backend](https://github.com/firelex/jeff/blob/f06788292874c21a5b5c41549ac220dd9e15da7f/src/jeff/mlx_backend.py), 2026-09 | 공개 코드 / 저자 보고 | code readout, full-weight training, MLX inference |
| R4 | Laurer et al., [Building Efficient Universal Classifiers with Natural Language Inference](https://arxiv.org/abs/2312.17543v1), 2023-12 | 연구 preprint | label-as-hypothesis scoring |
| R5 | Jared Palmer, [Kev](https://github.com/jaredpalmer/kev) / [model.py](https://github.com/jaredpalmer/kev/blob/main/kev/model.py), 2026-09 | 공개 구현 | candidate scoring과 adaptation |
| R6 | TianyuCodings, [NanoJev](https://github.com/TianyuCodings/NanoJev/tree/76fdfc9ecdca45a9bcef17991a07d3041a87685a) | 공개 구현 | candidate path와 decision head |
| R7 | Stepanov et al., [GLiClass: Generalist Lightweight Model for Sequence Classification Tasks](https://arxiv.org/abs/2508.07662v1), 2025-08-11 | 연구 preprint / [코드](https://github.com/Knowledgator/GLiClass) | joint labels/text, multi-label, architectural variants |
| R8 | Stepanov et al., [SCX Router: Streaming Zero-Shot Model Selection with a Decoder-KV Classifier and a Real-World Task Ontology](https://arxiv.org/abs/2609.02292v1), 2026-09-02 | 연구 preprint | decoder scorer, persistent text KV cache |
| R9 | Yaldiz et al., [Balancing Classification and Calibration Performance in Decision-Making LLMs via Calibration Aware Reinforcement Learning](https://arxiv.org/abs/2601.13284v1), 2026-01-19 | 연구 preprint | decision-token probability, SFT/RLVR tradeoff |
| R10 | Damani et al., [Beyond Binary Rewards: Training LMs to Reason About Their Uncertainty](https://arxiv.org/abs/2507.16806v1), 2025-07-22 | 연구 preprint | RLCR, proper scoring rewards |
| R11 | Kwok et al., [Contrastive Language Models: A System One Model for Fast and Generalizable Decision-Making](https://github.com/Contrastive-LM/CLM) / [fine-tuning code](https://github.com/Contrastive-LM/CLM/blob/main/train/finetune.py), 2026-09 | 저자 기술 보고 / 공개 코드 | contrastive states-actions, projection heads |
| R12 | mmastrac, [Structured reads on DiffusionGemma / djev](https://github.com/mmastrac/djev), 2026-09 | 공개 구현 | diffusion answer-slot readout |
| R13 | Guo et al., [On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html), ICML 2017 | 학술 논문 | temperature scaling |
| R14 | Saman Sarker Joy, [Do System One Decisions Add Up? A Study of Probabilistic Coherence](https://arxiv.org/abs/2609.33971v1), 2026-09-27 | 평가 preprint | 계층형 분류의 확률 일관성 |
| R15 | Hu et al., [JevAdvBench: A Benchmark and Black-Box Attacks for Reinforcement Learning for Calibrated Decisions Models](https://arxiv.org/abs/2609.31142v1), 2026-09-25 | 평가 preprint | re-run baseline, adversarial typed decisions |
| R16 | MLX-LM, [Fine-Tuning with LoRA or QLoRA](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md), [trainer](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/tuner/trainer.py), [Qwen3.5 model](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/models/qwen3_5.py) | 공식 라이브러리 문서 / 코드 | Mac training API와 구현 제약 |
| R17 | [Devlin et al., BERT, 2018](https://arxiv.org/abs/1810.04805v2) | 기반 연구 | pretrained representation + task classifier |
| R18 | [Yin, Hay & Roth, Benchmarking Zero-shot Text Classification, 2019](https://arxiv.org/abs/1909.00161v1) | 기반 연구 | NLI entailment 기반 text–label 분류 |
| R19 | [Zaratiana et al., GLiNER, 2023](https://arxiv.org/abs/2311.08526) | 기반 연구 | GLiClass가 명시한 label/text uni-encoder의 출발점 |
| R20 | [Reimers & Gurevych, Sentence-BERT, 2019](https://arxiv.org/abs/1908.10084v1) | 기반 연구 | siamese encoder, sentence similarity |
| R21 | [van den Oord et al., Contrastive Predictive Coding, 2018](https://arxiv.org/abs/1807.03748v2) | 기반 연구 | InfoNCE, positive/negative 구분 |
| R22 | [Austin et al., Structured Denoising Diffusion Models in Discrete State-Spaces, 2021](https://arxiv.org/abs/2107.03006) | 기반 연구 | discrete corruption와 denoising |
| R23 | [Nie et al., Large Language Diffusion Models (LLaDA), 2025](https://arxiv.org/abs/2502.09992) | 연구 preprint | masked-token diffusion LM의 사례 |
| R24 | [Hinton, Vinyals & Dean, Distilling the Knowledge in a Neural Network, 2015](https://arxiv.org/abs/1503.02531v1) | 기반 연구 | soft-target distillation, temperature |
| R25 | [Hu et al., LoRA: Low-Rank Adaptation of Large Language Models, 2021](https://arxiv.org/abs/2106.09685) | 기반 연구 | frozen weight + low-rank adaptation |
| R26 | [PyTorch, Start Locally](https://pytorch.org/get-started/locally/) | 공식 설치 문서 | CPU/CUDA/ROCm; [MPS backend](https://docs.pytorch.org/docs/stable/notes/mps.html) |
| R27 | [Hugging Face Transformers, Qwen3.5](https://huggingface.co/docs/transformers/model_doc/qwen3_5) | 공식 라이브러리 문서 | PyTorch causal LM 및 forward interface |
| R28 | [MLX, Build and Install](https://ml-explore.github.io/mlx/build/html/install.html) | 공식 설치 문서 | Apple Silicon / Linux CUDA / Linux CPU 지원 범위 |
| R29 | [denis-pplx, AutoJev](https://github.com/denis-pplx/autojev) | 공개 구현 | Jeff README에 명시된 직접 학습 코드 계보 |

2026년 자료도 발표 형식과 재현 상태가 다르다. GLiClass·SCX·calibration 논문은 읽을 수 있는 연구 자료이고, CLM·Jeff·Kev·djev는 각각 공개 범위가 다른 기술 보고·구현이다. 단일 제품명 아래 동일한 architecture나 학습 recipe로 묶지 않는다.

## 2. Jev like 모델 학습에 사용할 데이터셋

**다운로드 완료.** 원본 파일을 `.cache/jevlike-training/datasets/`에 저장했고, 고정 revision·원본 URL·SHA-256을 각 `manifest.json`에 기록했다. 이 디렉터리는 Git의 `.cache` 제외 규칙에 따라 저장소에 포함되지 않는다. 다른 환경에서는 `python scripts/download_decision_datasets.py`로 다시 내려받는다.

| 데이터 | 내려받은 원본 | 행 수 | 사용할 판단 |
| --- | --- | --- | --- |
| [BANKING77](https://github.com/PolyAI-LDN/task-specific-datasets/tree/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data) | train.csv / test.csv / categories.json / LICENSE | train 10,003 / test 3,080 | 단일 문의 의도 Choice, 77개 의도 |
| [GoEmotions simplified](https://huggingface.co/datasets/google-research-datasets/go_emotions/tree/add492243ff905527e67aeb8b80c082af02207c3/simplified) | train / validation / test Parquet, dataset card | train 43,410 / validation 5,426 / test 5,427 | 감정별 독립 Noul, 27개 감정 + neutral |
| [SST-5 sentence-level](https://huggingface.co/datasets/SetFit/sst5/tree/e51bdcd8cd3a30da231967c1a249ba59361279a3) | train / validation / test JSONL, dataset card | train 8,544 / validation 1,101 / test 2,210 | 감성 극성 Score, 순서가 있는 5단계 |

BANKING77는 원저자 GitHub의 CSV를 받아 최신 `datasets`에서 구식 dataset script를 실행하지 않도록 했다. GoEmotions는 `raw`의 개별 annotator 행 전체 대신 실습에 적합한 `simplified` 분할을 받았다. 세 데이터 모두 영어 데이터이며 한국어 성능의 근거가 되지 않는다.

**다음 작성 범위:** 데이터 preview, label 분포, 작은 균형 표본, 원본 ID·중복을 고려한 split, multi-label→Noul 변환, calibration fold. GoEmotions는 주관적 annotation을 가지므로 태그 없음이 객관적인 감정 부재를 보장한다는 가정도 점검한다. [BANKING77 dataset card](https://huggingface.co/datasets/PolyAI/banking77), [GoEmotions dataset card](https://huggingface.co/datasets/google-research-datasets/go_emotions)

In [ ]:
# 다운로드 현황만 확인한다. 학습·네트워크 요청을 실행하지 않는다.
from pathlib import Path
import json

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file()), Path.cwd())
DATA_ROOT = ROOT / ".cache" / "jevlike-training" / "datasets"
for name in ("banking77", "go_emotions", "sst5"):
    manifest_path = DATA_ROOT / name / "manifest.json"
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
        print(name, "revision:", manifest["revision"])
        print("files:", [entry["path"] for entry in manifest["files"]])
        print("bytes:", sum(entry["bytes"] for entry in manifest["files"]))
    else:
        print(name, "미다운로드: python scripts/download_decision_datasets.py")

### 2.1 Score 데이터셋: SST-5

**선정 이유.** SST(Stanford Sentiment Treebank)는 영화 리뷰 문장의 감성을 5단계로 구분한다. Socher et al., EMNLP 2013의 데이터이며, 여기서는 SetFit이 제공하는 **문장 단위** 버전을 사용한다. 구문 단위 phrase 데이터를 섞지 않는다. 작은 파일과 명시적인 순서형 정답이 있어 CPU·CUDA·MLX 공통 Score 실습에 적합하다. [원 논문](https://aclanthology.org/D13-1170/), [Stanford 원본](https://nlp.stanford.edu/sentiment/), [SetFit dataset card](https://huggingface.co/datasets/SetFit/sst5/blob/e51bdcd8cd3a30da231967c1a249ba59361279a3/README.md)

**Score 질문:** “이 영화 리뷰 문장의 감성 극성은 어느 수준인가?” `state`에는 원본 `text`를 넣는다. 아래 level ID는 파일의 `label`과 같으며, 원본 `label_text`와 대조한다.

| level / 학습 target | 원본 label_text | 척도 설명 |
| --- | --- | --- |
| 0 | very negative | 매우 부정적 |
| 1 | negative | 부정적 |
| 2 | neutral | 중립 |
| 3 | positive | 긍정적 |
| 4 | very positive | 매우 긍정적 |

모델은 이 순서의 5개 logits를 반환하고 softmax로 $p_0,\ldots,p_4$를 계산한다. 선택 level은 argmax, 기대 점수는 $E[v]=\sum_{k=0}^{4} k p_k$이다. `E[v]/4`로 0~1 범위에 표시할 수 있지만 **정답일 확률이나 confidence가 아니다**. 기대 점수에는 단계 간 간격을 동일하게 취급하는 가정이 들어간다. 원본은 범주 정답이며 연속 점수·확률 정답을 제공하지 않는다.

**학습·평가 계획.** 5개 level CE → 순서형 손실 비교. MAE는 두 단계 오차를 한 단계 오차보다 크게 계산하고 QWK는 순서형 일치도를 평가한다. probability calibration은 별도로 확인한다. 초기 실습은 train에서 level별 최대 64개, validation에서 level별 최대 16개를 고정 seed로 추출할 수 있다. 균형 표본은 동작 확인에 쓰고, 최종 평가에는 원본 test 분포를 사용한다. 기존 train / dev / test 구분을 유지하며 `dev.jsonl`만 로컬에서 `validation.jsonl`로 이름을 바꿨다. 중복 문장 검사는 모델 학습 전에 수행한다.

**해석 범위.** 영화 리뷰의 감성 극성을 학습하는 태스크다. 위험도·긴급도·일반 의사결정 Score로 전용하려면 해당 척도의 정답 데이터와 별도 평가가 필요하다. **라이선스:** 선택한 SetFit/sst5 저장소에는 dataset license가 명시되어 있지 않다. 임의로 Apache/MIT로 표기하지 않았으며 manifest에도 미명시 상태를 기록했다. 재배포·제품 사용 전에는 원 제공자의 조건을 확인한다.

아래 코드는 로컬 JSONL을 읽고 **이 노트북 내부의 학습 레코드**로 변환하는 예제다. 외부 Jev SDK 요청 형식이 아니다. 모델 호출·학습·다운로드는 하지 않으며 셀은 아직 실행하지 않았다.


In [ ]:
# SST-5 → Score 학습 레코드 예제 (stdlib만 사용, 미실행)
from collections import Counter
import random

SCORE_LEVELS = ["very negative", "negative", "neutral", "positive", "very positive"]
SCORE_QUESTION = "What is the sentiment polarity of this movie review sentence?"

def load_sst5(split):
    if split not in {"train", "validation", "test"}:
        raise ValueError("Unknown split")
    path = DATA_ROOT / "sst5" / f"{split}.jsonl"
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    for row in rows:
        assert 0 <= row["label"] < 5
        assert SCORE_LEVELS[row["label"]] == row["label_text"]
    return rows

def to_score_record(row):
    return {"state": row["text"], "primitive": "score",
            "question": SCORE_QUESTION, "levels": list(SCORE_LEVELS),
            "level_values": [0, 1, 2, 3, 4], "target_level": row["label"]}

def small_balanced_sample(rows, per_level=64, seed=42):
    rng = random.Random(seed)
    selected = []
    for level in range(5):
        pool = [row for row in rows if row["label"] == level]
        selected.extend(rng.sample(pool, min(per_level, len(pool))))
    rng.shuffle(selected)
    return selected

sst5_train = load_sst5("train")
score_train_preview = [to_score_record(row) for row in small_balanced_sample(sst5_train)]
print("Original train levels:", Counter(row["label"] for row in sst5_train))
print("Small train sample:", len(score_train_preview))
print(score_train_preview[0])
# validation은 load_sst5("validation")에서 별도로 추출하고 test는 원본 분포를 유지한다.


## 3. Jev like 모델을 학습하는 pipeline

*상세 구현 예정.* 1번 B 방식의 readout-only → LoRA+readout 순서로 **PyTorch CPU/CUDA와 MLX** 경로를 작성한다. 데이터 변환·split·code mapping·평가는 공유하고 모델 forward·gradient·optimizer·저장만 backend별로 구현한다. CPU는 작은 표본·짧은 입력·frozen backbone/head 학습으로 시작하며 GPU 환경에서는 같은 계약으로 범위를 확대한다.

Score 경로는 SST-5의 level 순서를 고정하고 5개 logits에 대한 CE로 시작한다. 평가에서는 accuracy, level MAE, QWK와 확률 보정을 함께 확인한다.

환경/패키지 확인 → tokenizer 확인 → 작은 forward/backward → 학습 → validation/calibration → 고정 test → checkpoint 재로드 순서로 구성한다. MPS/ROCm 및 Linux MLX는 필요한 연산 지원을 확인한 후 선택한다. 실제 성공·메모리·지연 시간은 실행한 환경의 결과로 기록한다.


## 4. Jev like 모델 경량화 (4, 8 bits quantization)

*상세 구현 예정.* 기준 정밀도와 8-bit / 4-bit 모델을 같은 test에서 비교한다. weight-only PTQ와 QLoRA 학습을 구분하고 adapter merge 순서·readout 정밀도·저장 형식을 기록한다. MLX 양자화와 PyTorch의 backend별 양자화는 라이브러리·kernel·지원 device·checkpoint 형식을 따로 선택한다. 같은 bit 수라는 이유로 파일 호환성이나 동일한 수치 결과를 가정하지 않는다.

환경에 맞는 구현이 지원되지 않으면 비양자화 모델로 배포 실습을 이어간다. 메모리뿐 아니라 logits·확률·calibration과 자동 처리 정책의 변화도 평가한다.


## 5. Jev like 모델 배포 및 사용

*상세 구현 예정.* 학습한 모델을 선택한 PyTorch 또는 MLX runtime에서 다시 로드하고 typed request → logits → 확률 → answer 경로를 연결한다. 로컬 API 계약과 Choice/Noul 예제는 공통으로 유지하며 model load·device·dtype·양자화 설정은 환경별로 구성한다. CPU 배포도 작은 요청으로 검증한다.

SST-5 기반 Score 인터페이스에는 5단계 감성 극성의 정의를 포함하고 지원 primitive와 실제로 검증한 능력을 명시한다. token 수·모델 실행 시간·HTTP 시간을 각각 기록한다. 공개 Jev/Jeff API 호출은 로컬 학습 경로와 별도 옵션으로 설명한다.
